### Needs `01-Prep-BaselinePipelineAB.ipynb` to be executed first

Reads from:
- `Baseline-PipelineA-PipelineB_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np

## Import Data

In [2]:
answers = pd.read_json("Baseline-PipelineA-PipelineB_ynorm.json")

CATEGORIES = ["value", "unit"]
VARIANTS = ["Base", "A", "B", "G"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    answers[col] = answers[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [4]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()

    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()

    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set

In [5]:
def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "Base" -> "A" -> "B" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,          # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "Base" -> "A" -> "B" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)

    return result

def eval(source, exact) -> pd.DataFrame:

    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)

    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)

    return gs_eval, in_place



def print_eval(gs_eval) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v:<4}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## YNORM

### Exact

In [6]:
ynorm_exact, ynorm_exact_extended = eval(answers, True)

print_eval(ynorm_exact)

### value ###
Base: True = 2073 || False = 135 || Quota = 93.89%
A   : True = 2100 || False = 108 || Quota = 95.11%
B   : True = 2078 || False = 130 || Quota = 94.11%
G   : True = 2124 || False = 84 || Quota = 96.2%

### unit ###
Base: True = 1983 || False = 225 || Quota = 89.81%
A   : True = 1967 || False = 241 || Quota = 89.09%
B   : True = 1964 || False = 244 || Quota = 88.95%
G   : True = 1995 || False = 213 || Quota = 90.35%



In [7]:
ynorm_exact_any, ynorm_exact_extended_any = eval(answers, False)

print_eval(ynorm_exact_any)

### value ###
Base: True = 2090 || False = 118 || Quota = 94.66%
A   : True = 2108 || False = 100 || Quota = 95.47%
B   : True = 2099 || False = 109 || Quota = 95.06%
G   : True = 2139 || False = 69 || Quota = 96.88%

### unit ###
Base: True = 1991 || False = 217 || Quota = 90.17%
A   : True = 1967 || False = 241 || Quota = 89.09%
B   : True = 1973 || False = 235 || Quota = 89.36%
G   : True = 1998 || False = 210 || Quota = 90.49%



In [8]:
ynorm_exact_extended.to_csv("Baseline-PipelineA-PipelineB-Eval.csv")

## By Scope

In [9]:
def print_eval_by_scope(gs_eval) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for scope in sorted(gs_eval["scope"].unique()):
            sub = gs_eval[gs_eval["scope"] == scope]
            print(f"  -- scope {scope} --")
            for v in VARIANTS:
                col = sub[f"{v}_{cat}_hit"]
                true_count  = col.value_counts().get(True, 0)
                false_count = col.value_counts().get(False, 0)
                quota       = round(col.mean() * 100, 2)
                print(f"  {v:<4}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

print_eval_by_scope(ynorm_exact_any)

### value ###
  -- scope 1 --
  Base: True = 542 || False = 19 || Quota = 96.61%
  A   : True = 543 || False = 18 || Quota = 96.79%
  B   : True = 544 || False = 17 || Quota = 96.97%
  G   : True = 547 || False = 14 || Quota = 97.5%
  -- scope 2lb --
  Base: True = 512 || False = 39 || Quota = 92.92%
  A   : True = 524 || False = 27 || Quota = 95.1%
  B   : True = 516 || False = 35 || Quota = 93.65%
  G   : True = 542 || False = 9 || Quota = 98.37%
  -- scope 2mb --
  Base: True = 517 || False = 28 || Quota = 94.86%
  A   : True = 525 || False = 20 || Quota = 96.33%
  B   : True = 521 || False = 24 || Quota = 95.6%
  G   : True = 533 || False = 12 || Quota = 97.8%
  -- scope 3 --
  Base: True = 519 || False = 32 || Quota = 94.19%
  A   : True = 516 || False = 35 || Quota = 93.65%
  B   : True = 518 || False = 33 || Quota = 94.01%
  G   : True = 517 || False = 34 || Quota = 93.83%

### unit ###
  -- scope 1 --
  Base: True = 494 || False = 67 || Quota = 88.06%
  A   : True = 478 || Fals

# Value-bearing recall
Recall = TP / (TP + FN) : In our case, that's the hit rate of an approach if the cell in the database has actually a value as well.

In [10]:

#  adaptation of print_eval()
def print_recall(extended) -> None:
    vb = extended[extended["value"].notna()]  # only cells that carry a gold figure
    print(f"(value-bearing cells: {len(vb)})")
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col         = vb[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            recall      = round(col.mean()*100,2) #Percent
            print(f"{v:<4}: True = {true_count} || False = {false_count} || Recall = {recall}%")
        print()

def print_recall_by_scope(extended) -> None:
    vb = extended[extended["value"].notna()]
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for scope in sorted(vb["scope"].unique()):
            sub = vb[vb["scope"] == scope]
            print(f"  -- scope {scope} ({len(sub)}) --")
            for v in VARIANTS:
                col         = sub[f"{v}_{cat}_hit"]
                true_count  = col.value_counts().get(True, 0)
                false_count = col.value_counts().get(False, 0)
                recall      = round(col.mean() * 100, 2)
                print(f"  {v:<4}: True = {true_count} || False = {false_count} || Recall = {recall}%")
        print()

In [11]:
print_recall(ynorm_exact_extended_any)

(value-bearing cells: 489)
### value ###
Base: True = 438 || False = 51 || Recall = 89.57%
A   : True = 451 || False = 38 || Recall = 92.23%
B   : True = 444 || False = 45 || Recall = 90.8%
G   : True = 455 || False = 34 || Recall = 93.05%

### unit ###
Base: True = 339 || False = 150 || Recall = 69.33%
A   : True = 310 || False = 179 || Recall = 63.39%
B   : True = 318 || False = 171 || Recall = 65.03%
G   : True = 314 || False = 175 || Recall = 64.21%



In [12]:
print_recall_by_scope(ynorm_exact_extended_any)

### value ###
  -- scope 1 (174) --
  Base: True = 168 || False = 6 || Recall = 96.55%
  A   : True = 168 || False = 6 || Recall = 96.55%
  B   : True = 169 || False = 5 || Recall = 97.13%
  G   : True = 165 || False = 9 || Recall = 94.83%
  -- scope 2lb (160) --
  Base: True = 127 || False = 33 || Recall = 79.38%
  A   : True = 139 || False = 21 || Recall = 86.88%
  B   : True = 132 || False = 28 || Recall = 82.5%
  G   : True = 153 || False = 7 || Recall = 95.62%
  -- scope 2mb (59) --
  Base: True = 53 || False = 6 || Recall = 89.83%
  A   : True = 54 || False = 5 || Recall = 91.53%
  B   : True = 54 || False = 5 || Recall = 91.53%
  G   : True = 51 || False = 8 || Recall = 86.44%
  -- scope 3 (96) --
  Base: True = 90 || False = 6 || Recall = 93.75%
  A   : True = 90 || False = 6 || Recall = 93.75%
  B   : True = 89 || False = 7 || Recall = 92.71%
  G   : True = 86 || False = 10 || Recall = 89.58%

### unit ###
  -- scope 1 (174) --
  Base: True = 120 || False = 54 || Recall = 68.9

## Alternative robustness result
Nine (9) reports have multiple values for a cell (report-year-scope combination) in the dataset for various reasons.
This quick analysis contrasts the recall from above against a further stripped down "gs_slim" excluding the nine reports.

In [13]:
multi_value_reports = [
    "Allianz_2022_report",
    "acuity brands inc_2022_report",
    "aixtron_2020_report",
    "allfunds group_2021_report",
    "granite construction inc_2020_report",
    "jetblue airways corp_2019_report",
    "jpmorgan chase_2020_report",
    "kilroy realty_2017_report",
    "uniper_2019_report",
]

for label, df in [("ANY", ynorm_exact_extended_any),
                     ("EXACT", ynorm_exact_extended)]:
    
    full = df[df["value"].notna()]
    cut  = full[~full["report_name"].isin(multi_value_reports)]

    table = pd.DataFrame({
        "variant": VARIANTS,
        "full_recall_pct": [round(full[f"{v}_value_hit"].mean() * 100, 2) for v in VARIANTS],
        "excl_multi_recall_pct": [round(cut[f"{v}_value_hit"].mean() * 100, 2) for v in VARIANTS],
    })
    # table.to_csv(f"Baseline-PipelineA-PipelineB-Alternative-robustness-{label.lower()}.csv", index=False)

    print(f"### value-bearing recall ({label})")
    print(table.to_string(index=False))
    print()
    print(f"n: full = {len(full)}")
    print(f"excluding multi = {len(cut)}")
    print()
    print()


### value-bearing recall (ANY)
variant  full_recall_pct  excl_multi_recall_pct
   Base            89.57                  92.62
      A            92.23                  94.81
      B            90.80                  94.54
      G            93.05                  94.54

n: full = 489
excluding multi = 366


### value-bearing recall (EXACT)
variant  full_recall_pct  excl_multi_recall_pct
   Base            86.09                  88.52
      A            90.59                  93.99
      B            86.50                  92.62
      G            89.98                  92.08

n: full = 489
excluding multi = 366




# Reports that are fully correct
Reports with all value-bearing cells correct

In [14]:
def print_report_full_match(extended) -> None:
    vb = extended[extended["value"].notna()]
    for v in VARIANTS:
        per_report = vb.groupby("report_name")[f"{v}_value_hit"].all()
        print(f"{v:<4}: {per_report.sum()}/{per_report.count()} ({round(per_report.mean()*100,1)}%)")

print_report_full_match(ynorm_exact_extended_any)

Base: 39/54 (72.2%)
A   : 41/54 (75.9%)
B   : 38/54 (70.4%)
G   : 44/54 (81.5%)


# Error Types
omission / substitution / addition

Explanation from the thesis:
* It is an *omission* error, when a value was present in the dataset, but the approach returned nothing.
* A *substitution* error, where the approach returned a different value.
* On empty dataset cells, an *addition* error, meaning the approach extracted a value the dataset does not list. 

In [15]:
def print_error_type(df) -> None:
    vb    = df[df["value"].notna()] # vb = vlaue-bearing
    empty = df[df["value"].isna()]

    # Checks if extraction returned a value and catches and empty list
    def returned_a_value(x):
        return isinstance(x, list) and len(x) > 0

    rows = []
    for v in VARIANTS:
        hit  = vb[f"{v}_value_hit"]
        miss = vb[~hit]
        rows.append({
            "variant": v,
            "hits": int(hit.sum()),
            "omission": int(miss[f"value_{v}"].apply(lambda x: not returned_a_value(x)).sum()),
            "substitution": int(miss[f"value_{v}"].apply(returned_a_value).sum()),
            "addition": int((~empty[f"{v}_value_hit"]).sum()),
        })

    table = pd.DataFrame(rows)
    # table.to_csv("Baseline-PipelineA-PipelineB-Error-Types.csv", index=False)

    print(table.to_string(index=False))

# membership frame gives the value-membership hits used everywhere else
print_error_type(ynorm_exact_extended_any)


variant  hits  omission  substitution  addition
   Base   438        29            22        67
      A   451        18            20        62
      B   444        27            18        64
      G   455        19            15        35
